# Serialización reproducible: del estimador al bundle

Notebook guiado para clase 1. Construiremos un bundle mínimo con `model.joblib` y `manifest.json`, sin importar módulos de solución.


## 0. Preparación

El modelo es simple y se entrena aquí para que la libreta sea ejecutable de arriba abajo en Databricks.


In [ ]:
import json
from pathlib import Path

import joblib
import pandas as pd
from sklearn.datasets import load_wine
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 42
WORK_DIR = Path('/dbfs/tmp/muiaap_s04_packaging')
WORK_DIR.mkdir(parents=True, exist_ok=True)

wine = load_wine(as_frame=True)
data = wine.frame.rename(columns={"target": "quality_class"})
FEATURES = list(wine.feature_names)
TARGET = "quality_class"
X_train, X_test, y_train, y_test = train_test_split(data[FEATURES], data[TARGET], test_size=0.2, random_state=SEED, stratify=data[TARGET])
model = Pipeline([("scale", StandardScaler()), ("clf", LogisticRegression(max_iter=500, random_state=SEED))])
model.fit(X_train, y_train)
print({"accuracy": model.score(X_test, y_test), "work_dir": str(WORK_DIR)})

## 1. Diseñar el manifiesto

**TODO 1:** decide qué metadatos mínimos necesita un consumidor. Importa porque `joblib` guarda bytes, pero no explica contrato ni versión. Inspecciona diccionarios JSON y verifica que el manifiesto tiene versión, features y métrica.


In [ ]:
manifest = {
    "model_name": "wine-classifier-demo",
    "model_version": "s04-demo-v1",
    "serialization_format": "joblib",
    "feature_names": FEATURES,
    "target_name": TARGET,
    "random_seed": SEED,
    "metrics": {"test_accuracy": float(model.score(X_test, y_test))},
}

required_keys = {"model_name", "model_version", "serialization_format", "feature_names", "target_name"}
assert required_keys.issubset(manifest)
print(json.dumps(manifest, indent=2)[:800])

## 2. Guardar el bundle

**TODO 2:** guarda modelo y manifiesto en la misma carpeta. Importa porque el despliegue debe mover una unidad completa. Inspecciona `joblib.dump` y `Path.write_text`. Verifica que ambos archivos existen.


In [ ]:
bundle_dir = WORK_DIR / manifest["model_version"]
bundle_dir.mkdir(parents=True, exist_ok=True)
model_path = bundle_dir / "model.joblib"
manifest_path = bundle_dir / "manifest.json"

joblib.dump(model, model_path)
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

assert model_path.is_file()
assert manifest_path.is_file()
print({"bundle_dir": str(bundle_dir)})

## 3. Cargar y validar antes de inferir

**TODO 3:** completa una validación más del manifiesto. Decide qué invariante te protegería de usar el modelo equivocado. Inspecciona `json.loads` y `assert`. Verifica que una copia dañada falla.


In [ ]:
def load_bundle(path: Path):
    loaded_manifest = json.loads((path / "manifest.json").read_text(encoding="utf-8"))
    loaded_model = joblib.load(path / "model.joblib")
    assert loaded_manifest["serialization_format"] == "joblib"
    assert loaded_manifest["feature_names"] == FEATURES
    assert loaded_manifest["target_name"] == TARGET
    # TODO 3: añade otra comprobación, por ejemplo model_version esperada.
    return loaded_model, loaded_manifest

loaded_model, loaded_manifest = load_bundle(bundle_dir)
sample = X_test.head(2)
predictions = loaded_model.predict(sample)
print({"predictions": predictions.tolist(), "version": loaded_manifest["model_version"]})
assert len(predictions) == 2

## 4. Simular un manifiesto roto

Antes de ejecutar, predice qué `assert` debe fallar si el orden de columnas no coincide.


In [ ]:
broken_dir = WORK_DIR / "broken_bundle"
broken_dir.mkdir(parents=True, exist_ok=True)
joblib.dump(model, broken_dir / "model.joblib")
broken_manifest = dict(manifest)
broken_manifest["feature_names"] = list(reversed(FEATURES))
(broken_dir / "manifest.json").write_text(json.dumps(broken_manifest, indent=2), encoding="utf-8")

try:
    load_bundle(broken_dir)
    print("ERROR: el bundle roto no falló")
except AssertionError:
    print("Correcto: la validación detectó el manifiesto incompatible")

## Cierre

La práctica de clase 2 parte de estas ideas: un artefacto de modelo no basta, el manifiesto documenta contrato y versión, y cargar un bundle debe validar antes de predecir.
